In [1]:
import pandas as pd
import numpy as np

In [2]:
# Chapter 10. Data Aggregation and Group Operations

In [3]:
# pandas provides a versatile .groupby() interface, enabling to slice and summarize datasets in a natural way
# query languages impose certain limitations on the kinds of group operations
# with the Python expressiveness can express as custom Python functions 

In [4]:
# 10.1 How to think About Group Operations

In [5]:
# group operations : split-apply-combine
# Series, DataFrame is split into groups based on one or more keys that you provide
# splitting is performed on a particular axis of an object
# once split, function is applied to each group, producing a new value
# results of all those function applications are combined into a result object

In [6]:
# each grouping key can take many forms, and the keys do not have to be all the same type:
# - A list or array of values that is the same length as the axis being grouped
# - A value indicating a column name in a DataFrame
# - A dictionary or Series giving a correspondence between the values on the axis being grouped and the group names
# - A function to be invoked on the axis index or the individuals labels in the index

In [7]:
df = pd.DataFrame({"key1":["a","a",None,"b","b","a",None],
                  "key2":pd.Series([1,2,1,2,1,None,1],dtype="Int64"),
                  "data1":np.random.standard_normal(7),
                  "data2":np.random.standard_normal(7)})
df

,key1,key2,data1,data2
0,a,1,-0.497585,-1.503506
1,a,2,-1.121651,-0.350777
2,None,1,0.675060,-0.841158
3,b,2,0.668479,0.070921
4,b,1,-3.165738,1.018356
5,a,<NA>,1.302965,0.654117
6,None,1,0.199632,-0.652003


In [8]:
# suppose you want to compute the mean of the data1 column using the labels from key1

In [9]:
# method 1 : access data1 and call groupby with the column (a Series) at key1:

grouped = df["data1"].groupby(df["key1"])
grouped

In [10]:
# grouped is now a special "GroupBy" object
# to compute group means we can call the GroupBy's .mean() method:
grouped.mean()

# here the data (a Series) has been aggregated by splitting the data on the group key;
# producing a new Series that is now indexed by the unique values in the key1 column
# the result index has the name "key1" because the DataFrame column df["key1"] did

key1
a   -0.105424
b   -1.248630
Name: data1, dtype: float64

In [11]:
# if instead passed multiple arrays as a list, we'd get something different:
means = df["data1"].groupby([df["key1"],df["key2"]]).mean()
means

key1  key2
a     1      -0.497585
      2      -1.121651
b     1      -3.165738
      2       0.668479
Name: data1, dtype: float64

In [12]:
# here we grouped the data using two keys, and the resulting Series now has a heirarchical index;
# which consist of the unique pairs of keys observed:
means.unstack()

key2,1,2
key1,,
a,-0.497585,-1.121651
b,-3.165738,0.668479


In [13]:
# in this example, the group keys are all Series, though they could be any arrays of the right lenght:
states = np.array(["OH","CA","CA","OH","OH","CA","OH"])
years = [2005,2005,2006,2005,2006,2005,2006]
df["data1"].groupby([states,years]).mean()

CA  2005    0.090657
    2006    0.675060
OH  2005    0.085447
    2006   -1.483053
Name: data1, dtype: float64

In [14]:
# frequently, the grouping information is the same DataFrame as the data you want to work on
# in this caes, you can pass column names as the group keys:
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.5,-0.105424,-0.400055
b,1.5,-1.248630,0.544639


In [15]:
df.groupby("key2").mean()

# there is no key1 column in the result since df["key1"] is not numeric data
# non-numeric data is said to be a nuisance column which is automatically excluded from result
# by default, all of the numeric columns are aggregated

TypeError: agg function failed [how->mean,dtype->object]

In [ ]:
df.groupby(["key1","key2"]).mean()

In [ ]:
# regardless of the objective in using .groupby(), generally useful GroupBy method is .size()
# .size() returns a Series containing group sizes:

df.groupby(["key1","key2"]).size()

In [ ]:
# any missing values in a group key are excluded from the result by default
# this behavior can be disabled by passing dropna=False to .groupby:

df.groupby("key1",dropna=False).size()

In [ ]:
df.groupby(["key1","key2"],dropna=False).size()

In [ ]:
# a group function similar to .size() is .count(), which computes the number of non-null values in each group:

df.groupby("key1").count()

In [ ]:
# Iterating over Groups
# the object returned by .groupby() supports iteration, generating a sequence of 2-tuples containing the group name along with the chunk of data

df

In [ ]:
for fuck,shit in df.groupby("key1"):
    print(fuck) # print the label, which is indexes under key1
    print(shit) # print the corresponding data to the label

# for loop above uses 2 variables - 1 for label and 1 for the corresponding data
# .groupby() requires 2 variables in for loop, since .groupby yields a tuple containing (Label, Data)

In [ ]:
# in the case of multiple keys, the first elemnt in the tuple will be a tuple of key values:
for(k1,k2),group in df.groupby(["key1","key2"]):
    print((k1,k2))
    print(group)

# for loop above : (k1,k2) and group are variables

In [ ]:
# you can choose to do whatever you want with the pieces of data
# computing a dictionary of the data pieces as a one-liner:
pieces = {name:group for name,group in df.groupby("key1")}
pieces["b"]

In [ ]:
# by default .groupby() groups on axis="index" but you can group on any other of the axes
# grouping the columns of the df above by whether they start with "key" or "data":

grouped = df.groupby({"key1":"key","key2":"key","data1":"data","data2":"data"},axis="columns")
grouped

In [ ]:
# we can print out the above groups like so:

for group_key,group_values in grouped:
    print(group_key)
    print(group_values)

In [ ]:
# Grouping with Dictionaries and Series

# grouping information may exist in a form other than an array

people = pd.DataFrame(np.random.standard_normal((5,5)),
                      columns=["a","b","c","d","e"],
                      index=["Joe","Steve","Wanda","Jill","Trey"])
people

In [ ]:
people.iloc[2:3,[1,2]]=np.nan # adding a few NaN values
people

In [ ]:
# suppose we get a group correspondence for the coluns and want to sum the columns by group:

mapping = {"a":"red","b":"red","c":"blue","d":"blue","e":"red","f":"orange"}
mapping

In [ ]:
# now, construct an array from this dictionary to pass to a .groupby()
# but instead we can just pass the dictionary (additional key "f" has been added to show unused grouping keys are okay)

by_column = people.groupby(mapping,axis="columns")
by_column

In [ ]:
by_column.sum()

# this is like conditional sum. By assiging grouping logic, given in dictionary, data can be grouped and aggregated.

In [ ]:
# the same functionality holds for Series, which can be viewed as a fixed-size mapping:

map_series = pd.Series(mapping)
map_series

In [ ]:
people.groupby(map_series,axis="columns").count()

In [ ]:
# Grouping with Functions

# defining functions for group mapping
# any function passed as a group key will be called once per index value with the return values being used as the group names
# suppose you want to group by name length - in this case passing a .len() works:

people.groupby(len).sum()

In [ ]:
# mixing functions with arrays, dictionaries, or Series is not a problem, as everything can be converted to arrays internally:

key_list = ["one","one","one","two","two"]
key_list

In [ ]:
people

In [ ]:
people.groupby([len,key_list]).min()

# grouping together: lenght of indexes & given key_list
# grouping results become tuples:
# [Joe,3,one] --> (3,"one")
# [Steve,5,one] --> (5,"one")
# [Wanda,5,one] --> (5,"one")
# [Jill,4,two] --> (4,"two")
# [Trey,4,two] --> (4,"two")

In [ ]:
# Grouping by Index Levels

# aggregating using one of the levels of an axis index

columns = pd.MultiIndex.from_arrays([["US","US","US","JP","JP"],
                                     [1,3,5,1,3]],
                                    names=["cty","tenor"])
columns

In [ ]:
hier_df = pd.DataFrame(np.random.standard_normal((4,5)),columns=columns) # note that calling columns it's the object columns, not "columns"
hier_df

In [ ]:
# to group by level, pass the level number or name using the "level" keyword:

hier_df.groupby(level="cty",axis="columns").count()

In [ ]:
# 10.2 Data Aggregation

# Aggregations refer to any data transformation that produces scalar values from arrays - mean, count, min, and sum

# Optimized common aggregations groupby methods
# .any(), .all() : Return True if any (one or more values) or all non-NA values are "truthy"
# .count() : Number of non-NA values
# .cummin(), .cummax() : Cumulative minimum and maximum of non-NA values
# .cumsum() : Cumulative sum of non-NA values
# .cumprod() : Cumulative product of non-NA values
# .first(), .last() : First and Last non-NA values
# .mean() : Mean of non-NA values
# .median() : Arithmetic median of non-NA values
# .min(), .max() : Minimum and Maximum of non-NA values
# .nth() : Retrieve value that would appear at position n with the data in sorted order
# .ohlc() : Compute four "open-high-low-close" statistics for time series-like data
# .prod() : Product of non-NA values
# .quantile() : Compute sample quantile
# .rank() : Ordinal ranks of non-NA values, like calling Series.rank()
# .size() : Compute group size, returning result as a Series
# .std(), .var() : Sample standard deviation and variance

In [17]:
# we can also use aggregrations of our own, even though not explicitly implemented for Groupby
# we can use .nsmallets()
# internally, Groupby slices up the Series, calls piece.nsmallest(n) for each piece, and then assembles those results into the result object:

df

,key1,key2,data1,data2
0,a,1,-0.497585,-1.503506
1,a,2,-1.121651,-0.350777
2,None,1,0.675060,-0.841158
3,b,2,0.668479,0.070921
4,b,1,-3.165738,1.018356
5,a,<NA>,1.302965,0.654117
6,None,1,0.199632,-0.652003


In [18]:
grouped = df.groupby("key1")
grouped

In [19]:
grouped["data1"].nsmallest(2) # .nsmallest() returns the first n rows with the smallest values in a specified column

# it can be like .nsmallest() == .sort_values() (Ascending) and head(n) combined together

key1   
a     1   -1.121651
      0   -0.497585
b     4   -3.165738
      3    0.668479
Name: data1, dtype: float64

In [20]:
# to use your own aggregation functions, pass any function that aggregates an array to the aggregate method or its short alias agg:

def peak_to_peak(arr):
    return arr.max() - arr.min()

In [21]:
grouped.agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,2.424616,2.157623
b,1,3.834216,0.947435


In [22]:
# you may notice that some methods, like .describe() also work, even though they are not aggregations:

grouped.describe()

key2                                           data1            ...  \
     count mean       std  min   25%  50%   75%  max count      mean  ...   
key1                                                                  ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0 -0.105424  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0 -1.248630  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     0.402690  1.302965   3.0 -0.400055  1.079655 -1.503506 -0.927141   
b    -0.290076  0.668479   2.0  0.544639  0.669938  0.070921  0.307780   

                                    
           50%       75%       max  
key1                                
a    -0.350777  0.151670  0.654117  
b     0.544639  0.781498  1.018356  

[2 rows x 24 columns]

In [23]:
# Note : Custom aggregation functions are generally much slower than the optimized functions 
#        this is because there is some extra overhead (function cells, data rearrangement) in constructing the intermediate group data chunks

In [24]:
# Column-Wise and Multiple Function Application

tips = pd.read_csv("examples/tips.csv")
tips.head()

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4


In [25]:
# add a tip_pct column with the tipe percentage of the total bill:

tips["tip_pct"] = tips["tip"]/tips["total_bill"]
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [26]:
# aggregating a Series or all of the columns of a DataFrame is a matter of using agg with desired function
# however, you may want to agg using a different function, depending on the column, or multiple functions at once
# first, group the tips by day and smoker:

grouped = tips.groupby(["day","smoker"])

In [27]:
# Note : for descriptive statistics, you can pass the name of the function as a string, within .agg()

grouped_pct = grouped["tip_pct"]

In [28]:
grouped_pct.agg("mean")

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

In [29]:
# if you pass a list of functions or function names instead, you get back a DataFrame with column names taken from the functions:

grouped_pct.agg(["mean","std",peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.151650  0.028123      0.067349
     Yes     0.174783  0.051293      0.159925
Sat  No      0.158048  0.039767      0.235193
     Yes     0.147906  0.061375      0.290095
Sun  No      0.160113  0.042347      0.193226
     Yes     0.187250  0.154134      0.644685
Thur No      0.160298  0.038774      0.193350
     Yes     0.163863  0.039389      0.151240

In [30]:
# you don't need to accept the names that GroupBy gives to the columns; notably, lambda functions have the name "<lambda>", which is hard to identify
# if you pass a list of (name,function) tuples, the first element of each tuple will be used as the DataFrame column names
# in tuple format .agg([("name","function"),("name","function")])

grouped_pct.agg([("average","mean"),("stdev",np.std)])

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_536\1529413272.py:5: FutureWarning: The provided callable <function std at 0x00000280BB5300E0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  grouped_pct.agg([("average","mean"),("stdev",np.std)])


average     stdev
day  smoker                    
Fri  No      0.151650  0.028123
     Yes     0.174783  0.051293
Sat  No      0.158048  0.039767
     Yes     0.147906  0.061375
Sun  No      0.160113  0.042347
     Yes     0.187250  0.154134
Thur No      0.160298  0.038774
     Yes     0.163863  0.039389

In [31]:
# with a DataFrame you have more options, as you can specify a list of functions to apply to all of the columns or different functions per column
# suppose we wanted to compute the same three statistics for the tip_pct and total_bill columns:

functions = ["count","mean","max"]

In [32]:
result = grouped[["tip_pct","total_bill"]].agg(functions)
result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

In [33]:
# the result DataFrame has hierarchical columns, the same you would get aggregating each column separately and using concat to glue the results together using the column names as the keys argument:
result["tip_pct"]

count      mean       max
day  smoker                           
Fri  No          4  0.151650  0.187735
     Yes        15  0.174783  0.263480
Sat  No         45  0.158048  0.291990
     Yes        42  0.147906  0.325733
Sun  No         57  0.160113  0.252672
     Yes        19  0.187250  0.710345
Thur No         45  0.160298  0.266312
     Yes        17  0.163863  0.241255

In [34]:
# a list of tuples with custom names can be passed:

ftuples = [("average","mean"),("Variance",np.var)]

In [35]:
grouped[["tip_pct","total_bill"]].agg(ftuples)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_536\2321675487.py:1: FutureWarning: The provided callable <function var at 0x00000280BB530220> is currently using SeriesGroupBy.var. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "var" instead.
  grouped[["tip_pct","total_bill"]].agg(ftuples)


tip_pct           total_bill            
              average  Variance    average    Variance
day  smoker                                           
Fri  No      0.151650  0.000791  18.420000   25.596333
     Yes     0.174783  0.002631  16.813333   82.562438
Sat  No      0.158048  0.001581  19.661778   79.908965
     Yes     0.147906  0.003767  21.276667  101.387535
Sun  No      0.160113  0.001793  20.506667   66.099980
     Yes     0.187250  0.023757  24.120000  109.046044
Thur No      0.160298  0.001503  17.113111   59.625081
     Yes     0.163863  0.001551  19.190588   69.808518

In [36]:
# suppose you wanted to apply potentially diferent funcitons to one or more of the columns
# pass a dictionary to .agg() that contains a mapping of column names to any of the function specifications listed so far:

grouped.agg({"tip":np.max,"size":"sum"})

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_536\2149908033.py:4: FutureWarning: The provided callable <function max at 0x00000280BB523560> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped.agg({"tip":np.max,"size":"sum"})


tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [37]:
grouped.agg({"tip_pct":["min","max","mean","std"],
             "size":"sum"})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

In [38]:
# a DataFrame will have hierarchical columns only if multiple funcitons are applied to at least one column

In [39]:
# Returning Aggregated Data Without Row Indexes

# you can disable the aggregated data coming with an index, in most cases, by passing "as_index=False" to .groupby():

tips

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
...,...,...,...,...,...,...,...
239,29.03,5.92,No,Sat,Dinner,3,0.203927
240,27.18,2.00,Yes,Sat,Dinner,2,0.073584
241,22.67,2.00,Yes,Sat,Dinner,2,0.088222
242,17.82,1.75,No,Sat,Dinner,2,0.098204


In [40]:
tips.groupby(["day","smoker"],as_index=False).mean(numeric_only=True)

# ** Pandas has updated and become more rigid. In older version, it would automatically drop text (string) columns
#     but with update, the automatic text drop process is gone, and need to explicitly specify "numeric_only=True"

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


In [41]:
# 10.3 Apply: General split-apply-combine

# the most general-purpose GroupBy method is .apply()
# .apply() splits the object being manipulated into pieces, invokes the passed function on each piece, and then attempts to concatenate the pieces


In [42]:
# tips dataset, suppose you wanted to select the top five tip_pct values by group
# first, write a function that selects the rows with the largest values in a particular column:

def top(df,n=5,column="tip_pct"):
    return df.sort_values(column,ascending=False)[:n]

In [43]:
top(tips,n=6)

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
232,11.61,3.39,No,Sat,Dinner,2,0.291990
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


In [44]:
# now, if we group by smoker, and cally apply with this function, we get the following:

tips.groupby("smoker").apply(top)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_536\2554940109.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby("smoker").apply(top)


total_bill   tip smoker   day    time  size   tip_pct
smoker                                                           
No     232       11.61  3.39     No   Sat  Dinner     2  0.291990
       149        7.51  2.00     No  Thur   Lunch     2  0.266312
       51        10.29  2.60     No   Sun  Dinner     2  0.252672
       185       20.69  5.00     No   Sun  Dinner     5  0.241663
       88        24.71  5.85     No  Thur   Lunch     2  0.236746
Yes    172        7.25  5.15    Yes   Sun  Dinner     2  0.710345
       178        9.60  4.00    Yes   Sun  Dinner     2  0.416667
       67         3.07  1.00    Yes   Sat  Dinner     1  0.325733
       183       23.17  6.50    Yes   Sun  Dinner     4  0.280535
       109       14.31  4.00    Yes   Sat  Dinner     2  0.279525

In [45]:
# to explain, first, the tips DataFrame is split into groups based on the value of smoker column.
# then the top function is called on each group, and the results of each function call are glued together using pd.concat(),
# labeling the pieces with the group names.
# the result therefore has a heirarchical index with an inner level that contains index values from the original DF

# if you pass a function to apply that takes other arguments or keywords, you can pass these after the function:

tips.groupby(["smoker","day"]).apply(top,n=1,column="total_bill")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_536\2373054506.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby(["smoker","day"]).apply(top,n=1,column="total_bill")


total_bill    tip smoker   day    time  size   tip_pct
smoker day                                                             
No     Fri  94        22.75   3.25     No   Fri  Dinner     2  0.142857
       Sat  212       48.33   9.00     No   Sat  Dinner     4  0.186220
       Sun  156       48.17   5.00     No   Sun  Dinner     6  0.103799
       Thur 142       41.19   5.00     No  Thur   Lunch     5  0.121389
Yes    Fri  95        40.17   4.73    Yes   Fri  Dinner     4  0.117750
       Sat  170       50.81  10.00    Yes   Sat  Dinner     3  0.196812
       Sun  182       45.35   3.50    Yes   Sun  Dinner     3  0.077178
       Thur 197       43.11   5.00    Yes  Thur   Lunch     4  0.115982

In [46]:
# what occurs inside the function passed is up to you; it must either return a pandas object or a scalar value
# here are various examples showing how to solve various problems using groupby

result = tips.groupby("smoker")["tip_pct"].describe()
result

,count,mean,std,min,25%,50%,75%,max
smoker,,,,,,,,
No,151.0,0.159328,0.039910,0.056797,0.136906,0.155625,0.185014,0.291990
Yes,93.0,0.163196,0.085119,0.035638,0.106771,0.153846,0.195059,0.710345


In [47]:
result.unstack("smoker")

       smoker
count  No        151.000000
       Yes        93.000000
mean   No          0.159328
       Yes         0.163196
std    No          0.039910
       Yes         0.085119
min    No          0.056797
       Yes         0.035638
25%    No          0.136906
       Yes         0.106771
50%    No          0.155625
       Yes         0.153846
75%    No          0.185014
       Yes         0.195059
max    No          0.291990
       Yes         0.710345
dtype: float64

In [48]:
# inside GroupBy, when you invoke a method like .describe(), it is actually just a shortcut for:

def f(group):
    return group.describe()

grouped.apply(f)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_536\973909651.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped.apply(f)


total_bill       tip  size   tip_pct
day  smoker                                            
Fri  No     count    4.000000  4.000000  4.00  4.000000
            mean    18.420000  2.812500  2.25  0.151650
            std      5.059282  0.898494  0.50  0.028123
            min     12.460000  1.500000  2.00  0.120385
            25%     15.100000  2.625000  2.00  0.137239
...                       ...       ...   ...       ...
Thur Yes    min     10.340000  2.000000  2.00  0.090014
            25%     13.510000  2.000000  2.00  0.148038
            50%     16.470000  2.560000  2.00  0.153846
            75%     19.810000  4.000000  2.00  0.194837
            max     43.110000  5.000000  4.00  0.241255

[64 rows x 4 columns]

In [49]:
# Suppressing the Group Keys

# earlier, you see that the resulting object has a hierarchical index formed from the group keys, along with the indexes of each piece of the original object
# you can disable this by passing group_keys=False to .groupby():

tips.groupby("smoker",group_keys=False).apply(top)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_536\3551738181.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby("smoker",group_keys=False).apply(top)


,total_bill,tip,smoker,day,time,size,tip_pct
232,11.61,3.39,No,Sat,Dinner,2,0.291990
149,7.51,2.00,No,Thur,Lunch,2,0.266312
51,10.29,2.60,No,Sun,Dinner,2,0.252672
185,20.69,5.00,No,Sun,Dinner,5,0.241663
88,24.71,5.85,No,Thur,Lunch,2,0.236746
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


In [50]:
# Quantile and Bucket Analysis

# from chapter 8, pd.cut() & pd.qcut() for slicing data up into buckets with bins of your choosing
# consider a simple random dataset and an equal-length bucket categorization using pd.cut():

frame = pd.DataFrame({"data1":np.random.standard_normal(1000),
                      "data2":np.random.standard_normal(1000)})
frame.head()

,data1,data2
0,-0.228490,-0.702944
1,-0.097333,0.603150
2,-0.901979,0.705301
3,-0.473914,1.494401
4,1.021452,-0.560148


In [51]:
quartiles = pd.cut(frame["data1"],4)
quartiles.head(10)

0    (-1.297, 0.229]
1    (-1.297, 0.229]
2    (-1.297, 0.229]
3    (-1.297, 0.229]
4     (0.229, 1.754]
5     (0.229, 1.754]
6    (-1.297, 0.229]
7    (-1.297, 0.229]
8    (-1.297, 0.229]
9    (-1.297, 0.229]
Name: data1, dtype: category
Categories (4, interval[float64, right]): [(-2.828, -1.297] < (-1.297, 0.229] < (0.229, 1.754] < (1.754, 3.279]]

In [52]:
# the Categorical object returned by cut can be passed directly to .groupby()
# so we could compute a set of group statistics for the qurartiles, like:

def get_stats(group):
    return pd.DataFrame(
        {"min":group.min(),"max":group.max(),
         "count":group.count(),"mean":group.mean()}
    )

In [53]:
grouped = frame.groupby(quartiles)
grouped

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_536\469962426.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = frame.groupby(quartiles)


In [54]:
grouped.apply(get_stats)

min       max  count      mean
data1                                                      
(-2.828, -1.297] data1 -2.822085 -1.303877    102 -1.696269
                 data2 -2.447539  1.773311    102 -0.128553
(-1.297, 0.229]  data1 -1.288432  0.228401    475 -0.411935
                 data2 -3.094822  3.132732    475  0.047724
(0.229, 1.754]   data1  0.229602  1.750702    377  0.817328
                 data2 -2.990357  2.297916    377 -0.037103
(1.754, 3.279]   data1  1.767901  3.279140     46  2.131799
                 data2 -3.364640  2.072374     46  0.059118

In [55]:
# keep in mind the same result could have been computed more simply with:

grouped.agg(["min","max","count","mean"])

data1                               data2            \
                       min       max count      mean       min       max   
data1                                                                      
(-2.828, -1.297] -2.822085 -1.303877   102 -1.696269 -2.447539  1.773311   
(-1.297, 0.229]  -1.288432  0.228401   475 -0.411935 -3.094822  3.132732   
(0.229, 1.754]    0.229602  1.750702   377  0.817328 -2.990357  2.297916   
(1.754, 3.279]    1.767901  3.279140    46  2.131799 -3.364640  2.072374   

                                  
                 count      mean  
data1                             
(-2.828, -1.297]   102 -0.128553  
(-1.297, 0.229]    475  0.047724  
(0.229, 1.754]     377 -0.037103  
(1.754, 3.279]      46  0.059118

In [56]:
# to compute equal-size buckets based on sample quantiles, use pd.qcut()
# we can pass 4 as the number of bucket compute sample quartiles, and pass labels=False to obtain just the quartile indices instead of intervals:

quartiles_samp = pd.qcut(frame["data1"],4,labels=False)
quartiles_samp.head()

0    1
1    1
2    0
3    1
4    3
Name: data1, dtype: int64

In [58]:
# Example: Filling Missing Values with Group-Specific Values

# cleaning up missing data involves .dropna() to drop or .fillna() to use fixed value
# here,filling null values with the mean:

s = pd.Series(np.random.standard_normal(6))
s

0   -0.299624
1   -1.027865
2   -0.829329
3    0.327463
4    0.483192
5    0.076865
dtype: float64

In [60]:
s[::2] = np.nan
s

0         NaN
1   -1.027865
2         NaN
3    0.327463
4         NaN
5    0.076865
dtype: float64

In [61]:
s.fillna(s.mean())

0   -0.207845
1   -1.027865
2   -0.207845
3    0.327463
4   -0.207845
5    0.076865
dtype: float64

In [62]:
# suppose you need to the fill value to vary by group
# one way to do this is to group the data and use .apply() with a function that calls fillna on each data chunk

states = ["Ohio","New York","Vermont","Florida","Oregon","Nevada","California","Idaho"]
group_key = ["East","East","East","East","West","West","West","West"]

In [65]:
data = pd.Series(np.random.standard_normal(8),index=states)
data

Ohio         -0.243587
New York     -0.134181
Vermont      -0.732574
Florida       2.121874
Oregon        0.490714
Nevada       -1.817152
California    0.704231
Idaho        -2.696773
dtype: float64

In [66]:
# setting some values to be NA:

data[["Vermont","Nevada","Idaho"]] = np.nan
data

Ohio         -0.243587
New York     -0.134181
Vermont            NaN
Florida       2.121874
Oregon        0.490714
Nevada             NaN
California    0.704231
Idaho              NaN
dtype: float64

In [67]:
data.groupby(group_key).size()

East    4
West    4
dtype: int64

In [68]:
data.groupby(group_key).count()

East    3
West    2
dtype: int64

In [69]:
data.groupby(group_key).mean()

East    0.581369
West    0.597472
dtype: float64

In [70]:
# we can fill the NA values using the group means, like:

def fill_mean(group):
    return group.fillna(group.mean())

In [71]:
data.groupby(group_key).apply(fill_mean)

East  Ohio         -0.243587
      New York     -0.134181
      Vermont       0.581369
      Florida       2.121874
West  Oregon        0.490714
      Nevada        0.597472
      California    0.704231
      Idaho         0.597472
dtype: float64

In [73]:
# in other case, you might have to predefine fill values in your code that vary by group
# since the groups have a name attribute set internally, we can use that:

fill_values = {"East":0.5,"West":-1}

def fill_func(group):
    return group.fillna(fill_values[group.name])

In [74]:
data.groupby(group_key).apply(fill_func)

East  Ohio         -0.243587
      New York     -0.134181
      Vermont       0.500000
      Florida       2.121874
West  Oregon        0.490714
      Nevada       -1.000000
      California    0.704231
      Idaho        -1.000000
dtype: float64

In [80]:
# Example: Random Sampling and Permutation

# suppose you want to draw a random sample (with or without replacement) from a large dataset for Monte Carlo simulation purpose
# there are a number of ways to perform the "draws"; here we use the .sample() method for Series - .sample() is used for random sampling
# here's a way to construct a deck of English-style playing cards:

suits = ["H","S","C","D"] # Hearts, Spades, Clubs, Diamonds

card_val = (list(range(1,11)) + [10]*3)*4
base_names = ["A"] + list(range(2,11)) + ["J","K","Q"]
cards = []

for suit in suits:
    cards.extend(str(num) + suit for num in base_names)

deck = pd.Series(card_val,index=cards)
deck.head(13)

AH      1
2H      2
3H      3
4H      4
5H      5
6H      6
7H      7
8H      8
9H      9
10H    10
JH     10
KH     10
QH     10
dtype: int64

In [82]:
# drawing a hand of five cards from the deck could be written as:

def draw(deck,n=5):
    return deck.sample(n)

In [83]:
draw(deck)

8D    8
7S    7
9S    9
5C    5
9D    9
dtype: int64

In [84]:
# suppose you wanted two random cards from each suit
# becuase the suit is the last character of each card name, we can group based on this and use .apply():

def get_suit(card):
    # last letter is suit
    return card[-1]

In [85]:
deck.groupby(get_suit).apply(draw,n=2)

C  8C     8
   KC    10
D  6D     6
   2D     2
H  2H     2
   KH    10
S  JS    10
   7S     7
dtype: int64

In [86]:
# alternatively, we could pass group_keys=False to drop the outer suit index, leaving in jsut the selected cards:

deck.groupby(get_suit,group_keys=False).apply(draw,n=2)

JC     10
5C      5
10D    10
JD     10
QH     10
4H      4
4S      4
7S      7
dtype: int64

In [87]:
# Example: Group Weighted Average and Correlation

# split-apply-combine --> .groupby()
# operations between columns in a DataFrame or two Series, such as a group weighted average are possible
# take this dataset containing group keys, values, and some weights:

df = pd.DataFrame({"category":["a","a","a","a","b","b","b","b"],
                   "data":np.random.standard_normal(8),
                   "weights":np.random.uniform(size=8)})
df

,category,data,weights
0,a,1.659464,0.554342
1,a,1.905144,0.945209
2,a,-1.047819,0.420246
3,a,-0.002086,0.972021
4,b,0.252475,0.669001
5,b,0.276758,0.332468
6,b,3.507938,0.332045
7,b,0.506615,0.942003


In [88]:
# the weighted average by category would then be:

grouped = df.groupby("category")

In [89]:
def get_wavg(group):
    return np.average(group["data"],weights=group["weights"])

In [90]:
grouped.apply(get_wavg)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_536\4286695940.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped.apply(get_wavg)


category
a    0.787843
b    0.836269
dtype: float64

In [91]:
# as another example, consider a financial dataset containing end-of-day stock prices and S&P 500 index (SPX):

close_px = pd.read_csv("examples/stock_px.csv",parse_dates=True,index_col=0)
close_px

,AAPL,MSFT,XOM,SPX
2003-01-02,7.40,21.11,29.22,909.03
2003-01-03,7.45,21.14,29.24,908.59
2003-01-06,7.45,21.52,29.96,929.01
2003-01-07,7.43,21.93,28.95,922.93
2003-01-08,7.28,21.31,28.83,909.93
...,...,...,...,...
2011-10-10,388.81,26.94,76.28,1194.89
2011-10-11,400.29,27.00,76.27,1195.54
2011-10-12,402.19,26.96,77.16,1207.25
2011-10-13,408.43,27.18,76.37,1203.66


In [92]:
close_px.info()

# DataFrame .info() method is a convenient way to get an overview of the contents of a DataFrame

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2214 entries, 2003-01-02 to 2011-10-14
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2214 non-null   float64
 1   MSFT    2214 non-null   float64
 2   XOM     2214 non-null   float64
 3   SPX     2214 non-null   float64
dtypes: float64(4)
memory usage: 86.5 KB


In [93]:
close_px.tail(4)

,AAPL,MSFT,XOM,SPX
2011-10-11,400.29,27.00,76.27,1195.54
2011-10-12,402.19,26.96,77.16,1207.25
2011-10-13,408.43,27.18,76.37,1203.66
2011-10-14,422.00,27.27,78.11,1224.58


In [96]:
# One task might be to copmute a DataFrame consisting of the yearly correlations of daily returns (computed from percent changes) with SPX
# as a way, first create a function that computes the pair-wise correlation of each column with the "SPX" column:

def spx_corr(group):
    return group.corrwith(group["SPX"])  # .corrwith() is used to compute the pairwise correlation between rows or columns of two DataFrame objects

In [95]:
# next, we group these percent changes by year, which can be extracted from each row label with a function that returns the year attribute of each datetime label:

def get_year(x):
    return x.year

In [97]:
# next, we copmute percentage change on close_px using .pct_change():

rets = close_px.pct_change().dropna() 
rets

# .pct_change() calculates the fractional change between the current and a prior element in Series or DF
# by default, it compares each row to the previous row

,AAPL,MSFT,XOM,SPX
2003-01-03,0.006757,0.001421,0.000684,-0.000484
2003-01-06,0.000000,0.017975,0.024624,0.022474
2003-01-07,-0.002685,0.019052,-0.033712,-0.006545
2003-01-08,-0.020188,-0.028272,-0.004145,-0.014086
2003-01-09,0.008242,0.029094,0.021159,0.019386
...,...,...,...,...
2011-10-10,0.051406,0.026286,0.036977,0.034125
2011-10-11,0.029526,0.002227,-0.000131,0.000544
2011-10-12,0.004747,-0.001481,0.011669,0.009795
2011-10-13,0.015515,0.008160,-0.010238,-0.002974


In [98]:
by_year = rets.groupby(get_year)

In [99]:
by_year.apply(spx_corr)

,AAPL,MSFT,XOM,SPX
2003,0.541124,0.745174,0.661265,1.0
2004,0.374283,0.588531,0.557742,1.0
2005,0.467540,0.562374,0.631010,1.0
2006,0.428267,0.406126,0.518514,1.0
2007,0.508118,0.658770,0.786264,1.0
2008,0.681434,0.804626,0.828303,1.0
2009,0.707103,0.654902,0.797921,1.0
2010,0.710105,0.730118,0.839057,1.0
2011,0.691931,0.800996,0.859975,1.0


In [100]:
# you could also compute intercolumn correlations
# here we compute the annual correlation between Apple and Microsoft:

def corr_aapl_msft(group):
    return group["AAPL"].corr(group["MSFT"])

In [101]:
by_year.apply(corr_aapl_msft)

2003    0.480868
2004    0.259024
2005    0.300093
2006    0.161735
2007    0.417738
2008    0.611901
2009    0.432738
2010    0.571946
2011    0.581987
dtype: float64

In [102]:
# Example: Group-WIse Linear Regression

# you can use .groupby() to perform more complex group-wise statistical analysis, as long as the function returns a pandas object or scalar value
# for example, can define the following .regress() function (using the statsmodels econometrics library)
# which executes an ordinary least squares (OLS) regression on each chunk of data:
# OLS is a method used in linear regression to estimate the parameters of a linear relationship between a dependent variable and one or more independent variables
# the goal of OLS is to minimize the sum of the squared differences between the observed values and the values predicted by the linear model
# simply put, OLS is the most common method that normally people consider "linear regression" is OLS

import statsmodels.api as sm

def regress(data,yvar=None,xvars=None):
    Y = data[yvar]
    X = data[xvars]
    X["intercept"] = 1. # make a column "intercept" for X
    result = sm.OLS(Y,X).fit()
    return result.params

In [103]:
# or, you can install statsmodels with conda if you don't have it already:

conda install statsmodels

SyntaxError: invalid syntax (1485672042.py, line 3)

In [104]:
# now, run a yearly linear regression of AAPL on SPX returns, execute: 

by_year.apply(regress,yvar="AAPL",xvars=["SPX"])

,SPX,intercept
2003,1.195406,0.000710
2004,1.363463,0.004201
2005,1.766415,0.003246
2006,1.645496,0.000080
2007,1.198761,0.003438
2008,0.968016,-0.001110
2009,0.879103,0.002954
2010,1.052608,0.001261
2011,0.806605,0.001514


In [105]:
# 10.4 Group Transform and "Unwrapped" GroupBys

# other than .apply(), there is another built-in method called .transform(), which is similar to .apply()
# but .transform() imposes more constraints on the kind of function you can use:
#  - it can produce a scalar value to be broadcast to the shape of the group
#  - it can produce an object of the same shape as the input group
#  - it must not mutate its input

In [106]:
# example

df = pd.DataFrame({'key':['a','b','c']*4,
                   'value':np.arange(12.)})
df

,key,value
0,a,0.0
1,b,1.0
2,c,2.0
3,a,3.0
4,b,4.0
5,c,5.0
6,a,6.0
7,b,7.0
8,c,8.0
9,a,9.0


In [107]:
# here are the group means by key:

g = df.groupby('key')['value']
g.mean()

key
a    4.5
b    5.5
c    6.5
Name: value, dtype: float64

In [109]:
# suppose instead we wanted to produce a Series of the same shape as df['value'] but with values replaced by the average grouped by 'key'
# we can pass a function that computes the mean of a single group to transform:

def get_mean(group):
    return group.mean()

g.transform(get_mean)

# the index is not showing but average of a = 4.5; b = 5.5; c = 6.5

0     4.5
1     5.5
2     6.5
3     4.5
4     5.5
5     6.5
6     4.5
7     5.5
8     6.5
9     4.5
10    5.5
11    6.5
Name: value, dtype: float64

In [110]:
# for built-in aggregation functions, we can pass a string alias as with the GroupBy .agg() method:

g.transform('mean')

0     4.5
1     5.5
2     6.5
3     4.5
4     5.5
5     6.5
6     4.5
7     5.5
8     6.5
9     4.5
10    5.5
11    6.5
Name: value, dtype: float64

In [111]:
# like .apply(), .transform() works with functions that return Series, but the result must be the same size as the input
# as an example, we can multiply each group by 2 using a helper function:

def times_two(group):
    return group * 2

g.transform(times_two)

0      0.0
1      2.0
2      4.0
3      6.0
4      8.0
5     10.0
6     12.0
7     14.0
8     16.0
9     18.0
10    20.0
11    22.0
Name: value, dtype: float64

In [112]:
# as a more complicated example, we can compute the ranks in descending order for each group:

def get_ranks(group):
    return group.rank(ascending=False)

g.transform(get_ranks)

0     4.0
1     4.0
2     4.0
3     3.0
4     3.0
5     3.0
6     2.0
7     2.0
8     2.0
9     1.0
10    1.0
11    1.0
Name: value, dtype: float64

In [113]:
# consider a group transformation function composed from simple aggregations:

def normalize(x):
    return(x - x.mean())/x.std()

# we can obtain equivalent results in this case using either transform or apply:

g.transform(normalize)

0    -1.161895
1    -1.161895
2    -1.161895
3    -0.387298
4    -0.387298
5    -0.387298
6     0.387298
7     0.387298
8     0.387298
9     1.161895
10    1.161895
11    1.161895
Name: value, dtype: float64

In [114]:
g.apply(normalize)

key    
a    0    -1.161895
     3    -0.387298
     6     0.387298
     9     1.161895
b    1    -1.161895
     4    -0.387298
     7     0.387298
     10    1.161895
c    2    -1.161895
     5    -0.387298
     8     0.387298
     11    1.161895
Name: value, dtype: float64

In [115]:
# built-in aggregate functions like 'mean' or 'sum' are often much faster than a general apply function.
# These also have a "fast path" when used with .transform(). This allows us to perform an "unwrapped" group operation:

g.transform('mean')

0     4.5
1     5.5
2     6.5
3     4.5
4     5.5
5     6.5
6     4.5
7     5.5
8     6.5
9     4.5
10    5.5
11    6.5
Name: value, dtype: float64

In [117]:
normalized = (df['value'] - g.transform('mean')) / g.transform('std')
normalized

0    -1.161895
1    -1.161895
2    -1.161895
3    -0.387298
4    -0.387298
5    -0.387298
6     0.387298
7     0.387298
8     0.387298
9     1.161895
10    1.161895
11    1.161895
Name: value, dtype: float64

In [118]:
# 10.5 Pivot Tables and Cross-Tabulation

# pivot table aggregates a table of data by one or more keys, arranging the data in a rectangle with some of the group keys along the rows and some along the columns.
# pivot table is possible in pandas through the groupby facility, combined with reshape operations utilizing hierarchical indexing
# DataFrame also has a .pivot_table() method, as a top level function.
# in addition to providing a convenience interface to groupby, .pivot_table() can add partial total, aslo know as "margins".

# using tipping dataset, suppose you wanted to compute a table of group means arranged by day and smoker on the rows:

tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [126]:
tips.pivot_table(index=["day","smoker"])

TypeError: agg function failed [how->mean,dtype->object]

In [125]:
# with pandas 2.0 and later, updated, including non-numeric values doesn't work any longer

# 'values' tells Pandas which columns to do the math on
# 'index' tells Pandas how to group the rows
tips.pivot_table(index=["day", "smoker"], values=["total_bill", "tip", "size", "tip_pct"])

size       tip   tip_pct  total_bill
day  smoker                                          
Fri  No      2.250000  2.812500  0.151650   18.420000
     Yes     2.066667  2.714000  0.174783   16.813333
Sat  No      2.555556  3.102889  0.158048   19.661778
     Yes     2.476190  2.875476  0.147906   21.276667
Sun  No      2.929825  3.167895  0.160113   20.506667
     Yes     2.578947  3.516842  0.187250   24.120000
Thur No      2.488889  2.673778  0.160298   17.113111
     Yes     2.352941  3.030000  0.163863   19.190588

In [127]:
# this could have been produced with .groupby() directly, using tips.groupby(["day","smoker"]).mean().
# suppose we want to take the average of only tip_pct and size, and additionally group by time.
# putting smoker in the table columns and time and day in the rows:

tips.pivot_table(index=["time","day"],columns="smoker",values=["tip_pct","size"])

size             tip_pct          
smoker             No       Yes        No       Yes
time   day                                         
Dinner Fri   2.000000  2.222222  0.139622  0.165347
       Sat   2.555556  2.476190  0.158048  0.147906
       Sun   2.929825  2.578947  0.160113  0.187250
       Thur  2.000000       NaN  0.159744       NaN
Lunch  Fri   3.000000  1.833333  0.187735  0.188937
       Thur  2.500000  2.352941  0.160311  0.163863

In [128]:
# we can augment this table to include partial totals by passing "margins=True".
# this has the effect of adding all row and column labels, with corresponding values being the group statistics for all the data within a single tier:

tips.pivot_table(index=["time","day"],columns="smoker",values=["tip_pct","size"],margins=True)

size                       tip_pct                    
smoker             No       Yes       All        No       Yes       All
time   day                                                             
Dinner Fri   2.000000  2.222222  2.166667  0.139622  0.165347  0.158916
       Sat   2.555556  2.476190  2.517241  0.158048  0.147906  0.153152
       Sun   2.929825  2.578947  2.842105  0.160113  0.187250  0.166897
       Thur  2.000000       NaN  2.000000  0.159744       NaN  0.159744
Lunch  Fri   3.000000  1.833333  2.000000  0.187735  0.188937  0.188765
       Thur  2.500000  2.352941  2.459016  0.160311  0.163863  0.161301
All          2.668874  2.408602  2.569672  0.159328  0.163196  0.160803

In [130]:
# here, the "All" values are means without taking into account smoker versus non-smoker (the All columns);
# or any of the two levels of grouping on the rows (the All row).

# to use an aggregation function other than mean, pass it to the aggfunc keyword argument.
# for example, "count" or len will give you a cross-tabulation (count or frequency) of group sizes 
# (though "count" will exclude null values from the count within data groups, while len will not):

# len thinks like a Physical Container. It includes everything—numbers, strings, and even null/missing values (NaN).
# count thinks like an Accountant. It will ignore/exclude any null values.

tips.pivot_table(index=["time","smoker"],columns="day",values="tip_pct",aggfunc=len,margins=True)

day             Fri   Sat   Sun  Thur  All
time   smoker                             
Dinner No       3.0  45.0  57.0   1.0  106
       Yes      9.0  42.0  19.0   NaN   70
Lunch  No       1.0   NaN   NaN  44.0   45
       Yes      6.0   NaN   NaN  17.0   23
All            19.0  87.0  76.0  62.0  244

In [131]:
# if some combinations are empty (or otherwise NA), you may wish to pass a fill_value:

tips.pivot_table(index=["time","size","smoker"],columns="day",values="tip_pct",fill_value=0)

day                      Fri       Sat       Sun      Thur
time   size smoker                                        
Dinner 1    No      0.000000  0.137931  0.000000  0.000000
            Yes     0.000000  0.325733  0.000000  0.000000
       2    No      0.139622  0.162705  0.168859  0.159744
            Yes     0.171297  0.148668  0.207893  0.000000
       3    No      0.000000  0.154661  0.152663  0.000000
            Yes     0.000000  0.144995  0.152660  0.000000
       4    No      0.000000  0.150096  0.148143  0.000000
            Yes     0.117750  0.124515  0.193370  0.000000
       5    No      0.000000  0.000000  0.206928  0.000000
            Yes     0.000000  0.106572  0.065660  0.000000
       6    No      0.000000  0.000000  0.103799  0.000000
Lunch  1    No      0.000000  0.000000  0.000000  0.181728
            Yes     0.223776  0.000000  0.000000  0.000000
       2    No      0.000000  0.000000  0.000000  0.166005
            Yes     0.181969  0.000000  0.000000  0.158843
       3    No      0.187735  0.000000  0.000000  0.084246
            Yes     0.000000  0.000000  0.000000  0.204952
       4    No      0.000000  0.000000  0.000000  0.138919
            Yes     0.000000  0.000000  0.000000  0.155410
       5    No      0.000000  0.000000  0.000000  0.121389
       6    No      0.000000  0.000000  0.000000  0.173706

In [132]:
# .pivot_table() options

# values : Column name or names to aggregate; by default, aggregates all numeric columns
# index : Column names or other group keys to group on the rows of the resulting pivot table
# columns : Column names or other group keys to group on the columns of the resulting pivot table
# aggfunc : Aggregation function or list of functions ("mean" by default); can be any function valid in a .groupby() context
# fill_value : Replacing missing values in the result table
# dropna : If True, do not include columns whose entries are all NA
# margins : Add row/column subtotals and grand total (False by default)
# margins_name : Name to use for the margin row/column labels when passing margins=True; defaults to "All"
# observed : With Categorical group keys, if True, show only the observed category values in the keys rather than all categories

In [139]:
# Cross-Tabulations: .crosstab()

# a cross-tabulation (or crosstab for short) is a special case of a pivot table that computes group frequencies.

from io import StringIO  # StringIO is a class found in the io module (Input/Output)

# it treats a regular string as if it were a file on the hard drive.
# StringIO(data) wraps the text in a "file-like" object so pandas can read i tjust like a real file.

In [140]:
data = """Sample Nationality Handedness
1 USA Right-handed
2 Japan Left-handed
3 USA Right-handed
4 Japan Right-handed
5 Japan Left-handed
6 Japan Right-handed
7 USA Right-handed
8 USA Left-handed
9 Japan Right-handed
10 USA Right-handed"""

data

'Sample Nationality Handedness\n1 USA Right-handed\n2 Japan Left-handed\n3 USA Right-handed\n4 Japan Right-handed\n5 Japan Left-handed\n6 Japan Right-handed\n7 USA Right-handed\n8 USA Left-handed\n9 Japan Right-handed\n10 USA Right-handed'

In [141]:
data = pd.read_table(StringIO(data),sep="\s+")  

# "\s" stands for whitespace (space, tab, or newline). "+" means "one or more".
# "split the columns whenever you see one or more spaces."

data

,Sample,Nationality,Handedness
0,1,USA,Right-handed
1,2,Japan,Left-handed
2,3,USA,Right-handed
3,4,Japan,Right-handed
4,5,Japan,Left-handed
5,6,Japan,Right-handed
6,7,USA,Right-handed
7,8,USA,Left-handed
8,9,Japan,Right-handed
9,10,USA,Right-handed


In [143]:
# as part of some survey analysis, we might want to summarize this data by nationality and handedness.
# .pivot_table() can be used, but the pd.crosstab() can be more convenient:

pd.crosstab(data["Nationality"],data["Handedness"],margins=True)

Handedness,Left-handed,Right-handed,All
Nationality,,,
Japan,2,3,5
USA,1,4,5
All,3,7,10


In [144]:
# the first two arguments to .crosstab() can each be an array or Series or a list of arrays. 
# as in the tips data:

pd.crosstab([tips["time"],tips["day"]],
            tips["smoker"],margins=True)

smoker        No  Yes  All
time   day                
Dinner Fri     3    9   12
       Sat    45   42   87
       Sun    57   19   76
       Thur    1    0    1
Lunch  Fri     1    6    7
       Thur   44   17   61
All          151   93  244